*** TO DO FOR CATAN: ***
RAINBOW: 
    1. vs Random
    2. vs Weighted Random
    3. vs MTCS
    4. vs Victory Point
    5. vs AlphaBeta
Masked PPO the same 
NFSP 
MuZero

MUZERO

In [ ]:
import copy
import sys


sys.path.append("../../../")

import torch

from agents.catan_player_wrapper import CatanPlayerWrapper

from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from modules.world_models.muzero_world_model import MuzeroWorldModel

from catanatron import Game, RandomPlayer, Color
from catanatron.players.mcts import MCTSPlayer
from catanatron.players.minimax import AlphaBetaPlayer
from catanatron.players.playouts import GreedyPlayoutsPlayer
from catanatron.players.search import VictoryPointPlayer
from catanatron.players.weighted_random import WeightedRandomPlayer
from catanatron.players.value import ValueFunctionPlayer
from utils.wrappers import record_video_wrapper
from stats.stats import StatTracker

from configs.games.catan import CatanConfig
from custom_gym_envs.envs.catan import (
    env as catan_env,
    CatanAECEnv,
)

from torch.optim import Adam, SGD

env = CatanConfig().make_env()
env = record_video_wrapper(copy.deepcopy(env), "./videos", 1)
params = {
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
    },
    # 1. Architecture (Robust Image processing via ResNet)
    "representation_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "dynamics_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "prediction_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    # 2. Heads (Distributional for stability)
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "categorical"},
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Architecture
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
        "residual_layers": [(24, 3, 1)],
    },
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # 3. Optimization
    # "learning_rate": 0.001,
    "optimizer": torch.optim.Adam,
    "weight_decay": 0.0001,
    "lr_ratio": float("inf"),  # Scale representation gradients
    "minibatch_size": 8,  # Scaled for stability
    "training_steps": 100000,
    # 4. Search / MCTS
    "num_simulations": 48,  # Increased for better planning
    "root_dirichlet_alpha": 0.3,
    "discount_factor": 0.999,  # Handle long sequences in Catan
    # 5. EfficientZero Features (Standard for robust MuZero)
    "reanalyze_ratio": 0.0,  # Maximize data efficiency
    "consistency_loss_factor": 0.0,
    # 6. Replay Buffer
    "replay_buffer_size": 10000,
    "min_replay_buffer_size": 32,
    "n_step": 250,
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "linear",
        "initial": 0.0,
        "final": 0.0,
        "decay_steps": 100000,
    },
    # 7. Misc
    "value_prefix": False,
    "record_video": True,
    "record_video_interval": 100,
    "num_workers": 4,
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "search_batch_size": 0,
    "use_virtual_mean": True,
    "search_backend": "aos",
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [20],  # after 5 moves in episode
        "values": [1.0, 0.0],  # start at 1.0, switch to 0.0 after step 5
    },
    "transfer_interval": 100,
    "gumbel": True,
}
game_config = CatanConfig()
config = MuZeroConfig(config_dict=params, game_config=game_config)


trainer = MuZeroTrainer(
    config=config,
    env=env,
    device=torch.device("cpu"),
    name="catan_bandit",
    stats=StatTracker("catan_bandit"),
    test_agents=[
        CatanPlayerWrapper(RandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(WeightedRandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(AlphaBetaPlayer, Color.BLUE),
    ],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 500
trainer.test_trials = 50

trainer.train()

In [ ]:
import copy
import sys


sys.path.append("../../../")

import torch

from agents.catan_player_wrapper import CatanPlayerWrapper

from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from modules.world_models.muzero_world_model import MuzeroWorldModel

from catanatron import Game, RandomPlayer, Color
from catanatron.players.mcts import MCTSPlayer
from catanatron.players.minimax import AlphaBetaPlayer
from catanatron.players.playouts import GreedyPlayoutsPlayer
from catanatron.players.search import VictoryPointPlayer
from catanatron.players.weighted_random import WeightedRandomPlayer
from catanatron.players.value import ValueFunctionPlayer
from utils.wrappers import record_video_wrapper
from stats.stats import StatTracker

from configs.games.catan import PlacementCatanConfig
from custom_gym_envs.envs.catan import (
    env as catan_env,
    CatanAECEnv,
)

from torch.optim import Adam, SGD

env = PlacementCatanConfig().make_env()
env = record_video_wrapper(copy.deepcopy(env), "./videos", 1)
params = {
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
    },
    # 1. Architecture (Robust Image processing via ResNet)
    "representation_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "dynamics_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "prediction_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    # 2. Heads (Distributional for stability)
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "categorical"},
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Architecture
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
        "residual_layers": [(24, 3, 1)],
    },
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # 3. Optimization
    # "learning_rate": 0.001,
    "optimizer": torch.optim.Adam,
    "weight_decay": 0.0001,
    "lr_ratio": float("inf"),  # Scale representation gradients
    "minibatch_size": 8,  # Scaled for stability
    "training_steps": 100000,
    # 4. Search / MCTS
    "num_simulations": 25,  # Increased for better planning
    "root_dirichlet_alpha": 0.3,
    "discount_factor": 0.999,  # Handle long sequences in Catan
    # 5. EfficientZero Features (Standard for robust MuZero)
    "reanalyze_ratio": 0.0,  # Maximize data efficiency
    "consistency_loss_factor": 0.0,
    # 6. Replay Buffer
    "replay_buffer_size": 10000,
    "min_replay_buffer_size": 32,
    "n_step": 9,
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "linear",
        "initial": 0.0,
        "final": 0.0,
        "decay_steps": 100000,
    },
    # 7. Misc
    "value_prefix": False,
    "record_video": True,
    "record_video_interval": 100,
    "num_workers": 4,
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "search_batch_size": 0,
    "use_virtual_mean": True,
    "search_backend": "python",
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [4],  # after 5 moves in episode
        "values": [1.0, 0.0],  # start at 1.0, switch to 0.0 after step 5
    },
    "transfer_interval": 100,
    "gumbel": True,
}
game_config = PlacementCatanConfig()
config = MuZeroConfig(config_dict=params, game_config=game_config)


trainer = MuZeroTrainer(
    config=config,
    env=env,
    device=torch.device("cpu"),
    name="catan_placement",
    stats=StatTracker("catan_placement"),
    test_agents=[
        CatanPlayerWrapper(RandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(WeightedRandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(AlphaBetaPlayer, Color.BLUE),
    ],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 500
trainer.test_trials = 50

trainer.train()

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


NameError: name 'get_actual_victory_points' is not defined

In [ ]:
import copy
import sys
import time
import numpy as np
import torch
from torch.optim import Adam

# Adjust paths as necessary
sys.path.append("../../../")

from catanatron.players.playouts import GreedyPlayoutsPlayer
from agents.catan_player_wrapper import CatanPlayerWrapper

from agents.learners.base import UniversalLearner
from modules.agent_nets.modular import ModularAgentNetwork

from configs.agents.muzero import MuZeroConfig
from configs.games.catan import CatanConfig
from custom_gym_envs.envs.catan import CatanAECEnv
from stats.stats import StatTracker

# ==============================================================================
# Buffer & Learning Components
# ==============================================================================
from replay_buffers.modular_buffer import ModularReplayBuffer, BufferConfig
from replay_buffers.writers import SharedCircularWriter
from replay_buffers.samplers import UniformSampler
from replay_buffers.transition import Transition

# Explicitly import the Loss and Target builder for Imitation Learning
from losses.losses import LossPipeline
from losses.losses import ImitationLoss
from agents.learners.target_builders import ImitationTargetBuilder

# ==============================================================================
# 1. Configuration & Architecture definition
# ==============================================================================
params = {
    # Architecture (Robust Image processing via ResNet)
    "representation_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "dynamics_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "prediction_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    # Heads
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "categorical"},
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        }
    },
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # Optimization
    "learning_rate": 0.001,
    "adam_epsilon": 1e-8,
    "optimizer": Adam,
    "weight_decay": 0.0001,
    "minibatch_size": 64,
    "replay_buffer_size": 10000,
    "min_replay_buffer_size": 64,
}

game_config = CatanConfig()
config = MuZeroConfig(config_dict=params, game_config=game_config)

# ==============================================================================
# 2. Environment, Expert, and Model Setup
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

env = game_config.make_env()

first_agent = env.possible_agents[0]
obs_dim = env.observation_space(first_agent).shape
num_actions = env.action_space(first_agent).n

agent_network = ModularAgentNetwork(
    config=config, input_shape=obs_dim, num_actions=num_actions
).to(device)

# ------------------------------------------------------------------------------
# NEW: Bypass MuZero Dynamics for Pure Root-Node Imitation Learning
# ------------------------------------------------------------------------------
from modules.world_models.inference_output import LearningOutput


def imitation_learner_inference(batch):
    """Bypasses sequence unrolling to only evaluate the root state."""
    observations = batch["observations"]

    # 1. Get the latent representation of the current state
    # Access via .representation as established in previous fix
    latent = agent_network.components["world_model"].representation(observations)

    # 2. Pass the latent state directly to the heads.
    # Use the correct key: "prediction_heads"
    head_preds = agent_network.components["prediction_heads"](latent)

    # 3. Return only what the ImitationLoss needs (the raw policy logits)
    # The head usually returns a dict with the key "policy"
    return LearningOutput(policies=head_preds.get("policy"))


# Override the default sequential inference with our root-only inference
agent_network.learner_inference = imitation_learner_inference
# ------------------------------------------------------------------------------

buffer_configs = [
    BufferConfig(name="observations", shape=obs_dim, dtype=torch.float32),
    BufferConfig(name="target_policies", shape=(num_actions,), dtype=torch.float32),
    BufferConfig(name="actions", shape=(), dtype=torch.long),
    BufferConfig(name="rewards", shape=(), dtype=torch.float32),
    BufferConfig(name="dones", shape=(), dtype=torch.bool),
    BufferConfig(name="legal_moves", shape=(num_actions,), dtype=torch.bool),
]

writer = SharedCircularWriter(max_size=config.replay_buffer_size)
sampler = UniformSampler()

replay_buffer = ModularReplayBuffer(
    max_size=config.replay_buffer_size,
    buffer_configs=buffer_configs,
    writer=writer,
    sampler=sampler,
)

# ------------------------------------------------------------------------------
# Instantiate the Target Builder & Loss Pipeline for Imitation Learning
# ------------------------------------------------------------------------------
target_builder = ImitationTargetBuilder()
loss_pipeline = LossPipeline([ImitationLoss(config, device, num_actions)])

learner = UniversalLearner(
    config=config,
    agent_network=agent_network,
    device=device,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=torch.float32,
    replay_buffer=replay_buffer,
    target_builder=target_builder,  # <-- Pass builder
    loss_pipeline=loss_pipeline,  # <-- Pass loss
    optimizer=Adam(
        agent_network.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        eps=config.adam_epsilon,
    ),
)

expert_agent = CatanPlayerWrapper(GreedyPlayoutsPlayer, None, num_playouts=3)

# ==============================================================================
# 3. Training Loop (Collect & Train)
# ==============================================================================
num_epochs = 50
episodes_per_epoch = 5
training_steps_per_epoch = 100

stats = StatTracker("catan_imitation")
stats._init_key("loss")
stats._init_key("accuracy")

print("Starting Imitation Learning test for UniversalLearner architecture...")

for epoch in range(num_epochs):
    # --- A. Data Collection ---
    print(f"\nEpoch {epoch+1}/{num_epochs} - Collecting data...")
    collected_transitions = 0

    for _ in range(episodes_per_epoch):
        env.reset()
        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()

            if termination or truncation:
                env.step(None)
                continue

            prediction_tuple = expert_agent.predict(obs, info, env)
            expert_action_tensor = expert_agent.select_actions(prediction_tuple, info)

            if hasattr(expert_action_tensor, "item"):
                expert_action = expert_action_tensor.item()
            else:
                expert_action = int(expert_action_tensor)

            target_policy = torch.zeros(num_actions)
            target_policy[expert_action] = 1.0

            # Format legal moves as a boolean mask matching the BufferConfig shape
            legal_moves_list = info.get("valid_actions", info.get("legal_moves", []))
            legal_moves_mask = np.zeros(num_actions, dtype=bool)
            if len(legal_moves_list) > 0:
                legal_moves_mask[legal_moves_list] = True

            data = {
                "observations": obs,
                "target_policies": (
                    target_policy.numpy()
                    if hasattr(target_policy, "numpy")
                    else target_policy
                ),
                "actions": expert_action,
                "rewards": reward,
                "dones": termination or truncation,
                "legal_moves": legal_moves_mask,
            }

            replay_buffer.store(**data)
            env.step(expert_action)
            collected_transitions += 1

    print(
        f"Collected {collected_transitions} transitions. Buffer size: {len(replay_buffer)}"
    )

    # --- B. Training ---
    if len(replay_buffer) > config.min_replay_buffer_size:
        print(f"Training for {training_steps_per_epoch} steps...")

        epoch_losses = []
        epoch_accuracies = []

        for _ in range(training_steps_per_epoch):
            # 1. Sample batch and train on UniversalLearner
            step_results = learner.step(stats=stats)
            if step_results and step_results.loss is not None:
                epoch_losses.append(step_results.loss.item())

            # 2. Compute explicit Accuracy (We sample again here for quick logging)
            raw_batch = replay_buffer.sample()
            batch = {
                k: v.to(device) if hasattr(v, "to") else v for k, v in raw_batch.items()
            }

            with torch.no_grad():
                inf_out = agent_network.obs_inference(batch["observations"])
                predictions = inf_out.policy.probs
                predicted_actions = predictions.argmax(dim=-1)
                target_actions = batch["target_policies"].argmax(dim=-1)
                acc = (predicted_actions == target_actions).float().mean().item()
                epoch_accuracies.append(acc)

        avg_loss = np.mean(epoch_losses) if epoch_losses else float("inf")
        avg_acc = np.mean(epoch_accuracies) if epoch_accuracies else 0.0

        print(f"--> Loss: {avg_loss:.4f} | Accuracy: {avg_acc*100:.2f}%")

print("\nFinished Testing.")

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using default results_path                  : results
Using default training_steps                : 10000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using default clipnorm                      : 0
Using         optimizer                     : <class 'torch.optim.adam.Adam'>
Using         weight_decay                  : 0.0001
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 128
Using         replay_buffer_size            : 10000
Using         min_replay_buffer_size        : 128
Using default n_step                        : 1
Using default discount_fact

KeyError: 'heads'